## upgrade molecules 

In [1]:

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem import rdChemReactions
from rdkit.Chem.MolStandardize import rdMolStandardize

def num_hdonors(mol):
    return rdMolDescriptors.CalcNumHBD(mol)

def sanitize(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None

# 1) Tautomerize and pick the one with max HBD (neutral transformation)
taut_enum = rdMolStandardize.TautomerEnumerator()
def pick_tautomer_max_hbd(mol):
    try:
        tautomers = taut_enum.Enumerate(mol)
    except Exception:
        return mol, num_hdonors(mol), 'orig'
    best = mol
    best_hbd = num_hdonors(mol)
    tag = 'orig'
    for t in tautomers:
        hbd = num_hdonors(t)
        if hbd > best_hbd:
            best, best_hbd = t, hbd
            tag = 'taut'
    return best, best_hbd, tag

import random
def upgrade_donor_poor_smiles(smiles_list, target_fraction=1.0, max_delta_atoms=1, max_new_mw=1):
    """
    - Only attempts to modify donor-poor molecules (HBD==0).
    - Works on a random subset (target_fraction) to avoid over-shifting the distribution.
    - Drops modifications that grow too much (atoms or MW).
    """
    out = []
    cont_total = 0
    cont_tau = 0
    for smi in tqdm(smiles_list):
        mol0 = sanitize(smi)
        if mol0 is None:
            out.append(smi)
            continue
        hbd_0 = num_hdonors(mol0)
        if num_hdonors(mol0) > 3 or random.random() > target_fraction:
            out.append(smi)
            continue

        mol_new, hbd_new, tag = pick_tautomer_max_hbd(mol0)
        smi_new = Chem.MolToSmiles(mol_new)
        mol_new = sanitize(smi_new)
        if mol_new is None:
            out.append(smi)
            continue

        cont_total += 1
        # Conservative guards
        if tag == 'taut':
            cont_tau += 1
        #print(tag, hbd_0, hbd_new)
        if (mol_new.GetNumAtoms() - mol0.GetNumAtoms()) > max_delta_atoms:
            print(f"Too many atoms: {mol_new.GetNumAtoms() - mol0.GetNumAtoms()}")
            #out.append(smi); continue
        if (Descriptors.MolWt(mol_new) - Descriptors.MolWt(mol0)) > max_new_mw:
            print(f"Too much weight: {Descriptors.MolWt(mol_new) - Descriptors.MolWt(mol0)}")
            #out.append(smi); continue

        out.append(smi_new)
    return out, cont_total, cont_tau

In [2]:
# Utilities: score all tautomers of a SMILES with the time model and pick the best
import os, sys
import torch
import numpy as np
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from torch_geometric.data import Data

# Ensure project root is on sys.path so we can import project modules
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("."), "..")))

from lib_functions.models import GINETimePredictor_MorganFP

# Local helpers from this folder
from lib_functions.data_preparation_utils import mol_to_graph
from lib_functions.data_preparation_utils import compute_features_cero_fps
# Time model class


def _build_time_model_input_from_mol(mol: Chem.Mol, device: torch.device) -> Data:
    """Build the PyG Data object for the time model from an RDKit Mol.
    Uses the same feature pipeline as in sample_molecules_FPSmodel.py via compute_features_cero_fps.
    """
    g = mol_to_graph(mol)
    if g is None:
        return None
    # Compute all features (graph, node, distances, edge index/attr, DOSD, fingerprint)
    ruido, gemb, nemb, distances, edge_index, edge_attr, natoms, _num, dosd_positions, fingerprint = compute_features_cero_fps(g)

    # Convert to tensors on device
    x = nemb.to(device)
    xA = gemb.to(device)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)
    distances = torch.tensor(distances, device=device, dtype=torch.float32)
    dosd_distances = torch.tensor(dosd_positions, device=device, dtype=torch.float32)
    morgan_fp = torch.tensor(fingerprint, device=device, dtype=torch.float32).unsqueeze(0)

    # Single-graph batch vector
    batch_vec = torch.zeros(x.size(0), dtype=torch.long, device=device)

    d = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        xA=xA,
        distances=distances,
        dosd_distances=dosd_distances,
        morgan_fp=morgan_fp,
        batch=batch_vec,
    )
    return d


def load_time_model(checkpoint_path: str, device: torch.device = None) -> GINETimePredictor_MorganFP:
    """Load the time model with weights and set to eval mode."""
    model = GINETimePredictor_MorganFP()
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model


def score_mol_time(mol: Chem.Mol, time_model: GINETimePredictor_MorganFP) -> float:
    """Return the scalar time prediction for a single RDKit Mol using the time model."""
    device = next(time_model.parameters()).device
    try:
        data = _build_time_model_input_from_mol(mol, device)
        if data is None:
            return float('inf')
        with torch.no_grad():
            pred = time_model(data)
            # pred shape [1, 1]; detach to CPU scalar
            return float(pred.squeeze().detach().cpu().item())
    except Exception:
        return float('inf')


def pick_tautomer_min_time(mol: str, time_model: GINETimePredictor_MorganFP):
    """Enumerate tautomers and pick the one with minimum predicted time.
    Returns (best_smiles, best_time, num_tautomers).
    """
    taut_enum = rdMolStandardize.TautomerEnumerator()
    try:
        tautomers = list(taut_enum.Enumerate(mol))
    except Exception:
        tautomers = [mol]

    if not tautomers:
        tautomers = [mol]

    orig_smi = Chem.MolToSmiles(mol, isomericSmiles=True)
    best_mol = mol
    best_score = score_mol_time(mol, time_model)
    tag = 'orig'
    orig_hbd = num_hdonors(mol)
    for t in tautomers:
        num_hbd = num_hdonors(t)
        if orig_hbd >= num_hbd:
            continue    
        # Ensure valid & sanitized
        smi_t = Chem.MolToSmiles(t, isomericSmiles=True)
        mt = Chem.MolFromSmiles(smi_t)
        if mt is None:
            continue
        try:
            Chem.SanitizeMol(mt)
        except Exception:
            continue
        score = score_mol_time(mt, time_model)
        if score < best_score and smi_t != orig_smi:
            #print(score, best_score)
            #print(smi_t, Chem.MolToSmiles(best_mol, isomericSmiles=True))
            best_score = score
            best_mol = mt
            tag = 'taut'

    if best_mol is None:
        return smiles, float('inf'),  'orig'

    return Chem.MolToSmiles(best_mol, isomericSmiles=True), best_score,  tag


def pick_tautomer_min_time_then_hbd(smiles: str, time_model: GINETimePredictor_MorganFP, rdkit_calc_hbd):
    """Pick the min-time tautomer, then (optionally) prefer the one with higher HBD among equal-time ties.
    rdkit_calc_hbd: function like rdMolDescriptors.CalcNumHBD.
    """
    smi_best, score_best, _ = pick_tautomer_min_time(smiles, time_model)
    return smi_best, score_best

def upgrade_donor_poor_smiles_time(smiles_list, time_model, target_fraction=1.0, max_delta_atoms=1, max_new_mw=1):
    """
    - Only attempts to modify donor-poor molecules (HBD==0).
    - Works on a random subset (target_fraction) to avoid over-shifting the distribution.
    - Drops modifications that grow too much (atoms or MW).
    """
    # Load your time model once
    
    out = []
    cont_total = 0
    cont_tau = 0
    hbd_diff = []
    for smi in tqdm(smiles_list):
        mol0 = sanitize(smi)
        if mol0 is None:
            out.append(smi)
            hbd_diff.append(0)
            continue
        hbd_0 = num_hdonors(mol0)
        if num_hdonors(mol0) > 3 or random.random() > target_fraction:
            out.append(smi)
            hbd_diff.append(0)
            continue

        smi_new, best_score, tag = pick_tautomer_min_time(mol0, time_model)
        mol_new = sanitize(smi_new)
        hbd_new = num_hdonors(mol_new)
        if mol_new is None:
            out.append(smi)
            hbd_diff.append(0)
            continue

        cont_total += 1
        # Conservative guards
        if tag == 'taut':
            cont_tau += 1
            hbd_diff.append(hbd_new - hbd_0)
        #print(tag, hbd_0, hbd_new)
        if (mol_new.GetNumAtoms() - mol0.GetNumAtoms()) > max_delta_atoms:
            print(f"Too many atoms: {mol_new.GetNumAtoms() - mol0.GetNumAtoms()}")
            #out.append(smi); continue
        if (Descriptors.MolWt(mol_new) - Descriptors.MolWt(mol0)) > max_new_mw:
            print(f"Too much weight: {Descriptors.MolWt(mol_new) - Descriptors.MolWt(mol0)}")
            #out.append(smi); continue

        out.append(smi_new)
    return out, cont_total, cont_tau, hbd_diff



Running on cuda
Running on cuda


## Processing of PubChem data

In [4]:
# read the file ../Data/CID-SMILES 
import tqdm 
with open('../Data/CID-SMILES-filtered-lt70.txt', 'r') as f:
    contador = 0
    smiles_list = []
    for line in tqdm.tqdm(f):
        smiles_list.append(line)
        contador += 1

print(len(smiles_list))

96498593it [00:13, 6985016.99it/s]

96498593


In [ ]:
# make 5 random samples of 1M smiles in CID-SMILES-filtered-lt70_x.txt
import random
for i in range(5):
    with open(f'../Data/CID-SMILES-filtered-lt70_{i}_nuevo3.txt', 'w') as f:
        for smile in random.sample(smiles_list, 1000000):
            f.write(smile)


In [1]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/250209_database_allmolecules_main_2_22_sinfps_timepred_2_22_sinfps_sinexplicit/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997400
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9974}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.951200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08849821049169743, 'MolLogP': 0.023844608853042785, 'MolWt': 0.04405876137545656, 'TPSA': 0.006250598701655778, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9976}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.950845
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09384478337780323, 'MolLogP': 0.024921112450060096, 'MolWt': 0.048664564898576984, 'TPSA': 0.0063951701207582866, 'NumHA

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997500
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9975}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.959227
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0953862307854017, 'MolLogP': 0.028427646124980617, 'MolWt': 0.049978487027541416, 'TPSA': 0.0067543417804071925, 'NumHAc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997500
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9975}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.959582
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09438857828954364, 'MolLogP': 0.02386117847362608, 'MolWt': 0.04795547035701375, 'TPSA': 0.007507988528775113, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997400
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9974}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.960074
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09360891522015481, 'MolLogP': 0.02577751244133056, 'MolWt': 0.04732917248826716, 'TPSA': 0.008974073080261168, 'NumHAcce

In [ ]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/250211_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997900
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9979}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.957846
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08208552826832065, 'MolLogP': 0.022480511821416874, 'MolWt': 0.044067466102722316, 'TPSA': 0.006827556322496198, 'NumHAc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997800
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9978}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.958623
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0870708788673, 'MolLogP': 0.023014811185750994, 'MolWt': 0.04860416392875263, 'TPSA': 0.0065334409463227985, 'NumHAccept

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9983}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964969
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08779398681275832, 'MolLogP': 0.02696346263327319, 'MolWt': 0.04994965559761199, 'TPSA': 0.00694889013124916, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9983}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.965051
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0867929874592782, 'MolLogP': 0.022592467538747767, 'MolWt': 0.04799060385170413, 'TPSA': 0.007601213905475487, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998400
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9984}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964613
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08868008066431439, 'MolLogP': 0.024694782409665588, 'MolWt': 0.047262663374660514, 'TPSA': 0.008816436797655964, 'NumHAc

In [11]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/250211_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_old.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_old.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9982}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.965727
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08768305187104429, 'MolLogP': 0.025320211518836545, 'MolWt': 0.047287987575331675, 'TPSA': 0.007203539729819245, 'NumHAc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_old.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997900
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9979}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964632
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0871572267369794, 'MolLogP': 0.024576902357118943, 'MolWt': 0.046325089214649826, 'TPSA': 0.007353222251063388, 'NumHAcc

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998100
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9981}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.965887
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08123077782967118, 'MolLogP': 0.023590999020049054, 'MolWt': 0.04453149343726291, 'TPSA': 0.008476382543551833, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_old.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9982}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.957254
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08359064899716137, 'MolLogP': 0.021245478855415147, 'MolWt': 0.04762436526875379, 'TPSA': 0.009256511348945432, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_old.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998800
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9988}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964617
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09035955647544924, 'MolLogP': 0.020104574619869656, 'MolWt': 0.044131624634868506, 'TPSA': 0.007098068064943521, 'NumHAc

In [1]:
from typing import List
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from rdkit import Chem

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/defog_final_smiles_strict.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        smiles_list.append(line.strip())

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 20000/20000 [00:01<00:00, 12094.33it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902700
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9027}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.940644
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.17234590676010675, 'MolLogP': 0.02416218606812271, 'MolWt': 0.13334472046894214, 'TPSA': 0.06288676314864727, 'NumHAccept

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902300
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9023}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.938544
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18168057392685164, 'MolLogP': 0.021950127566052538, 'MolWt': 0.14484822612311657, 'TPSA': 0.06486671560060088, 'NumHAccep

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902400
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9024}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.936478
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18327657234931635, 'MolLogP': 0.028102653894884096, 'MolWt': 0.14500586125083345, 'TPSA': 0.07071146103826031, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902300
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9023}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.938750
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.17927704460997623, 'MolLogP': 0.027796606108705723, 'MolWt': 0.13710503699843596, 'TPSA': 0.06982212445896813, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9022}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.936113
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.1841077869469833, 'MolLogP': 0.032350057997562305, 'MolWt': 0.14482295219232402, 'TPSA': 0.07463109253095687, 'NumHAccept

In [2]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger

class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/defog_final_smiles_relaxed.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        smiles_list.append(line.strip())

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 20000/20000 [00:01<00:00, 11308.17it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9837}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.939296
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.16861906670250182, 'MolLogP': 0.023626875213526716, 'MolWt': 0.11787134087418136, 'TPSA': 0.07773832061679789, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9832}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.937772
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.1779850677816635, 'MolLogP': 0.02101336874401846, 'MolWt': 0.12874739397518775, 'TPSA': 0.07918174912551573, 'NumHAccepto

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983400
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9834}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.936703
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.1793024362386052, 'MolLogP': 0.027356626651588535, 'MolWt': 0.1291427530758776, 'TPSA': 0.08543101347801241, 'NumHAccepto

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9832}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.937509
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.17512359942312716, 'MolLogP': 0.02733554340264757, 'MolWt': 0.12130690572923651, 'TPSA': 0.08497114998368183, 'NumHAccept

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983000
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9830}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.933996
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18043950074478105, 'MolLogP': 0.03206734953414035, 'MolWt': 0.1289615126263839, 'TPSA': 0.08940802733495204, 'NumHAccepto

In [3]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/digress_generated_all.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        smiles_list.append(line.strip())

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 18000/18000 [00:01<00:00, 11976.33it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833400
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8334}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.919377
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.25282699284392646, 'MolLogP': 0.02394502412593921, 'MolWt': 0.10677369147195212, 'TPSA': 0.06342683282654621, 'NumHAccept

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833100
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8331}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.916533
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.2612587448592874, 'MolLogP': 0.026265045488391293, 'MolWt': 0.11716381465342146, 'TPSA': 0.06562157430763146, 'NumHAccept

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8332}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.914738
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.2627201560461523, 'MolLogP': 0.030071211153376354, 'MolWt': 0.11832488037289933, 'TPSA': 0.07000395818651473, 'NumHAccept

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8332}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.915195
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.26140273287612387, 'MolLogP': 0.026242246434818282, 'MolWt': 0.10876639268796999, 'TPSA': 0.0705408937505126, 'NumHAccept

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833400
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8334}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.912153
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.26391022375143497, 'MolLogP': 0.03005443855486541, 'MolWt': 0.11925625383468721, 'TPSA': 0.07399809984720658, 'NumHAccept

In [5]:
from typing import List
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from rdkit import Chem

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/gdss_zinc250k-sample.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            smiles = parts[0]
            validity = int(parts[1])
            if validity == 0:  # valid molecule
                smiles_list.append(smiles)
            else:  # invalid molecule
                smiles_list.append("None")

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



  0%|          | 0/20000 [00:00<?, ?it/s]

100%|██████████| 20000/20000 [00:01<00:00, 16209.79it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.957400
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9574}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.940800
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9408}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.693838
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.5903106757666947, 'MolLogP': 0.051594085554260684, 'MolWt': 0.8323249309809903, 'TPSA': 0.46007200452576014, 'NumHAccepto

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.957400
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9574}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.941200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9412}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.698429
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.574313654530361, 'MolLogP': 0.06462221976305299, 'MolWt': 0.8597744566680425, 'TPSA': 0.4561561191179085, 'NumHAcceptors'

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.957400
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9574}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.941500
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9415}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.707608
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.5729246635450882, 'MolLogP': 0.06299426298632954, 'MolWt': 0.8333938035989565, 'TPSA': 0.45099625765784174, 'NumHAcceptor

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.957400
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9574}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.941700
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9417}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.698699
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.5813761121777519, 'MolLogP': 0.06256124189718654, 'MolWt': 0.8208963731566842, 'TPSA': 0.45136522757698144, 'NumHAcceptor

In [6]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/GRUM_zinc250k.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            smiles = parts[0]
            validity = int(parts[1])
            if validity == 0:  # valid molecule
                smiles_list.append(smiles)
            else:  # invalid molecule
                smiles_list.append("None")

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 13000/13000 [00:00<00:00, 13342.26it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9847}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.982700
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9827}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.870699
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.1513989935324424, 'MolLogP': 0.34843830893739774, 'MolWt': 0.3591135836473267, 'TPSA': 0.12412356535026685, 'NumHAcceptor

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9847}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983000
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9830}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.865125
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.14758078990381163, 'MolLogP': 0.4084124412856035, 'MolWt': 0.3322890164493814, 'TPSA': 0.1295467207130222, 'NumHAcceptors

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9847}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.982500
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9825}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.867722
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.1477735846881099, 'MolLogP': 0.397124943862787, 'MolWt': 0.3711758075474564, 'TPSA': 0.14187895691799407, 'NumHAcceptors'

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9847}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9832}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.874596
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.14523781395665408, 'MolLogP': 0.37782894746316664, 'MolWt': 0.343307086911884, 'TPSA': 0.12473662198001607, 'NumHAcceptor

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9847}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9832}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.865256
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.14789739498494975, 'MolLogP': 0.4220929186511587, 'MolWt': 0.3514183280905855, 'TPSA': 0.14054413152357875, 'NumHAcceptor

In [7]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger
from tqdm import tqdm
from rdkit import Chem
import random


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/CDMOL_guacamol_smiles.smiles', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        smiles_list.append(line.strip())

if len(smiles_list) <10000:
   for i in range(10000-len(smiles_list)):
      smiles_list.append("None")

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")

if len(filtered_smiles) <10000:
    # randomly duplicate some smiles to reach 10000
    for i in range(10000-len(filtered_smiles)):
        filtered_smiles.append(random.choice(filtered_smiles))

print(len(filtered_smiles))

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 10000/10000 [00:00<00:00, 11864.55it/s]
INFO : Benchmarking distribution learning, version v2


10000
Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.751100
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7511}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.954602
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0645296057774216, 'MolLogP': 0.018110347258852157, 'MolWt': 0.058213703122524066, 'TPSA': 0.009800515806120225, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.750600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7506}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.953490
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0727838117864628, 'MolLogP': 0.021453153502832306, 'MolWt': 0.06360387047859188, 'TPSA': 0.0107538432071926, 'NumHAccepto

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.750600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7506}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.954554
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.07098576301780021, 'MolLogP': 0.025978802693960603, 'MolWt': 0.06841757611217736, 'TPSA': 0.012319110961034973, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.750200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7502}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.952493
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.06753700795574506, 'MolLogP': 0.023857996696075523, 'MolWt': 0.06012689008254953, 'TPSA': 0.012840464585161132, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.750800
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7508}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.952089
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.07477862317027886, 'MolLogP': 0.02551667934319582, 'MolWt': 0.06493923598569591, 'TPSA': 0.013270252933356819, 'NumHAccep

In [11]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger
from tqdm import tqdm
from rdkit import Chem
import random


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/jtvae_generated_all_231108.txt', 'r') as f:
    smiles_list = [line.strip() for line in f]


if len(smiles_list) <10000:
   for i in range(10000-len(smiles_list)):
      smiles_list.append("None")

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")

if len(filtered_smiles) <10000:
    # randomly duplicate some smiles to reach 10000
    for i in range(10000-len(filtered_smiles)):
        filtered_smiles.append(random.choice(filtered_smiles))

print(len(filtered_smiles))

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 30000/30000 [00:02<00:00, 14982.49it/s]
INFO : Benchmarking distribution learning, version v2


30000
Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997500
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9975}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.556121
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.7655214712815506, 'MolLogP': 1.174521974769803, 'MolWt': 5.053613886844112, 'TPSA': 0.28085815610832754, 'NumHAcceptors'

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9972}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.561696
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.7700810479715963, 'MolLogP': 1.2127376924648126, 'MolWt': 5.094274730973369, 'TPSA': 0.29371407637067387, 'NumHAcceptors

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997400
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9974}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.562479
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.7355538008921831, 'MolLogP': 1.2283743247808816, 'MolWt': 5.058522844814536, 'TPSA': 0.3000882282396541, 'NumHAcceptors'

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997300
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9973}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.555136
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.7380450598817534, 'MolLogP': 1.2027941091977616, 'MolWt': 5.035148885859033, 'TPSA': 0.29103175364764555, 'NumHAcceptors

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.996800
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9968}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.557806
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.7486206136389006, 'MolLogP': 1.2806249836761632, 'MolWt': 5.121334407668356, 'TPSA': 0.3103725075422282, 'NumHAcceptors'

# Time model hablation

In [8]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/251010_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit_notime/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998400
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9984}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.934466
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.07779759759458463, 'MolLogP': 0.008201523121135096, 'MolWt': 0.04208934340541194, 'TPSA': 0.005881704410401613, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999300
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9993}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.935777
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08305042059480348, 'MolLogP': 0.010278493954220806, 'MolWt': 0.04655813449698315, 'TPSA': 0.006082885741025872, 'NumHAcc

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999000
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9990}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.947390
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08447995507281322, 'MolLogP': 0.008787756157443288, 'MolWt': 0.04989954529186349, 'TPSA': 0.005346642001847121, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998700
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9987}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.948981
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08363498758870253, 'MolLogP': 0.008272063183929066, 'MolWt': 0.045897599718851566, 'TPSA': 0.006526720906377457, 'NumHAc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999500
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9995}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.947891
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0820673317860279, 'MolLogP': 0.009266736072018635, 'MolWt': 0.04552953697964043, 'TPSA': 0.007502391480499944, 'NumHAcce

In [10]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/251012_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit_unseed2/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9976}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.947109
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08061347679870218, 'MolLogP': 0.017567539175497242, 'MolWt': 0.042467812804104534, 'TPSA': 0.005521250685798362, 'NumHAc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997500
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9975}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.947550
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08802732527350321, 'MolLogP': 0.021710940393574456, 'MolWt': 0.04669692241989309, 'TPSA': 0.005701443243016632, 'NumHAcc

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997900
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9979}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.956121
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.087329615085227, 'MolLogP': 0.021460022667787508, 'MolWt': 0.05016108565162849, 'TPSA': 0.005824998784334137, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997800
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9978}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.956983
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08541881485323945, 'MolLogP': 0.021952591395208568, 'MolWt': 0.04561740165548756, 'TPSA': 0.006058444159676096, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9982}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.956889
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09039022079499912, 'MolLogP': 0.017717164720614076, 'MolWt': 0.04684222323031142, 'TPSA': 0.007762550725167922, 'NumHAcc

# Novelty whole PubChem

In [ ]:
# read the file ../Data/CID-SMILES 
import tqdm 
with open('../Data/CID-SMILES-filtered-lt70.txt', 'r') as f:
    contador = 0
    smiles_list_pc = []
    for line in tqdm.tqdm(f):
        smiles_list_pc.append(line)
        contador += 1

smiles_csv = '../mols_gen/250211_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit/all_generated_molecules.csv'
smiles_list_gen = pd.read_csv(smiles_csv).smiles.to_list()
